# Сверка ЧОД и Фин.Рез (Excel vs DRP)

Этот ноутбук сравнивает значения `ЧОД` и `Фин.Рез` между тремя Excel-отчетами (январь, февраль, март) и таблицей DRP `sbx_da.tmp_datamart_q1`.

Что проверяется:
- агрегаты по месяцам;
- расхождения по ИНН;
- вклад разных типов причин (`only_excel`, `only_drp`, `value_mismatch`).

In [ ]:
import re
from getpass import getpass

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [ ]:
# Параметры Excel (каждый файл соответствует своему месяцу)
excel_sources = [
    {
        'report_month': '2026-01-01',
        'path': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx',
        'header': 0,
    },
    {
        'report_month': '2026-02-01',
        'path': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx',
        'header': 0,
    },
    {
        'report_month': '2026-03-01',
        'path': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx',
        'header': 0,
    },
]

# Названия колонок в Excel
excel_col_inn = 'ИНН'
excel_col_chod = 'ЧОД'
excel_col_finrez = 'Фин.Рез'

# DRP источник
drp_schema = 'sbx_da'
drp_table = 'tmp_datamart_q1'

In [ ]:
def normalize_inn(value):
    if pd.isna(value):
        return None
    s = re.sub(r'\D+', '', str(value))
    return s if s else None


def parse_numeric(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    if not s:
        return np.nan

    s = s.replace('\xa0', '').replace(' ', '')
    if ',' in s and '.' in s:
        # Если есть оба разделителя, считаем, что точка - тысячные, запятая - десятичная
        s = s.replace('.', '').replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '.')

    s = re.sub(r'[^0-9\.-]', '', s)
    if s in {'', '-', '.', '-.'}:
        return np.nan

    try:
        return float(s)
    except Exception:
        return np.nan


def month_start(value):
    dt = pd.to_datetime(value, errors='coerce')
    if pd.isna(dt):
        return pd.NaT
    return pd.Timestamp(dt.year, dt.month, 1)

In [ ]:
# Загрузка и нормализация Excel
excel_frames = []
for src in excel_sources:
    report_month = month_start(src['report_month'])
    path = src['path']
    header = src.get('header', 0)

    df = pd.read_excel(path, header=header)

    required = [excel_col_inn, excel_col_chod, excel_col_finrez]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"В файле {path} отсутствуют колонки: {missing}")

    cur = pd.DataFrame({
        'report_month': report_month,
        'inn': df[excel_col_inn].apply(normalize_inn),
        'chod': df[excel_col_chod].apply(parse_numeric),
        'fin_result': df[excel_col_finrez].apply(parse_numeric),
    })
    cur['source_file'] = path
    excel_frames.append(cur)

excel_raw = pd.concat(excel_frames, ignore_index=True)
excel_raw = excel_raw[excel_raw['report_month'].notna()]

excel_qc = (
    excel_raw.groupby('report_month', as_index=False)
    .agg(
        rows=('inn', 'size'),
        inn_cnt=('inn', 'nunique'),
        chod_sum=('chod', 'sum'),
        finrez_sum=('fin_result', 'sum'),
    )
    .sort_values('report_month')
)

display(excel_qc)

In [ ]:
# Агрегация Excel до уровня месяц + ИНН
excel_by_inn = (
    excel_raw.groupby(['report_month', 'inn'], as_index=False)
    .agg(
        excel_chod=('chod', 'sum'),
        excel_finrez=('fin_result', 'sum'),
    )
)

excel_by_month = (
    excel_by_inn.groupby('report_month', as_index=False)
    .agg(
        excel_rows=('inn', 'size'),
        excel_inn_cnt=('inn', 'nunique'),
        excel_chod_sum=('excel_chod', 'sum'),
        excel_finrez_sum=('excel_finrez', 'sum'),
    )
)

display(excel_by_month.sort_values('report_month'))

In [ ]:
# Загрузка данных из DRP
months_sql = ', '.join(
    [f"'{pd.to_datetime(x['report_month']).strftime('%Y-%m-%d')}'" for x in excel_sources]
)

target_fq = f"{drp_schema}.{drp_table}"

sql_drp = f"""
select
    report_month,
    inn,
    chod,
    fin_result
from {target_fq}
where cast(report_month as date) in ({months_sql})
"""

drp_user = input('DRP user: ').strip()
drp_password = getpass('DRP password: ')

drp_conn = connect(
    'drp',
    settings={
        'user_name': drp_user,
        'password': drp_password,
        'sslmode': 'require',
    },
)

with drp_conn:
    drp_raw = drp_conn.fetch(sql_drp)

drp_raw.columns = [c.lower() for c in drp_raw.columns]

drp_raw['report_month'] = pd.to_datetime(drp_raw['report_month'], errors='coerce').dt.to_period('M').dt.to_timestamp()
drp_raw['inn'] = drp_raw['inn'].apply(normalize_inn)
drp_raw['chod_num'] = drp_raw['chod'].apply(parse_numeric)
drp_raw['finrez_num'] = drp_raw['fin_result'].apply(parse_numeric)

drp_by_inn = (
    drp_raw.groupby(['report_month', 'inn'], as_index=False)
    .agg(
        drp_chod=('chod_num', 'sum'),
        drp_finrez=('finrez_num', 'sum'),
    )
)

drp_by_month = (
    drp_by_inn.groupby('report_month', as_index=False)
    .agg(
        drp_rows=('inn', 'size'),
        drp_inn_cnt=('inn', 'nunique'),
        drp_chod_sum=('drp_chod', 'sum'),
        drp_finrez_sum=('drp_finrez', 'sum'),
    )
)

display(drp_by_month.sort_values('report_month'))

In [ ]:
# Таблица A: сравнение агрегатов по месяцам
summary_a = (
    excel_by_month.merge(drp_by_month, on='report_month', how='outer')
    .sort_values('report_month')
    .reset_index(drop=True)
)

for col in [
    'excel_rows', 'excel_inn_cnt', 'drp_rows', 'drp_inn_cnt',
    'excel_chod_sum', 'excel_finrez_sum', 'drp_chod_sum', 'drp_finrez_sum'
]:
    if col in summary_a.columns:
        summary_a[col] = summary_a[col].fillna(0)

summary_a['diff_chod_abs'] = summary_a['drp_chod_sum'] - summary_a['excel_chod_sum']
summary_a['diff_finrez_abs'] = summary_a['drp_finrez_sum'] - summary_a['excel_finrez_sum']

summary_a['diff_chod_pct'] = np.where(
    summary_a['excel_chod_sum'] != 0,
    summary_a['diff_chod_abs'] / summary_a['excel_chod_sum'] * 100,
    np.nan,
)
summary_a['diff_finrez_pct'] = np.where(
    summary_a['excel_finrez_sum'] != 0,
    summary_a['diff_finrez_abs'] / summary_a['excel_finrez_sum'] * 100,
    np.nan,
)

display(summary_a)

In [ ]:
# Таблица B: расхождения по ИНН
comparison_b = (
    excel_by_inn.merge(
        drp_by_inn,
        on=['report_month', 'inn'],
        how='outer',
        indicator=True,
    )
)

comparison_b['excel_chod'] = comparison_b['excel_chod'].fillna(0.0)
comparison_b['excel_finrez'] = comparison_b['excel_finrez'].fillna(0.0)
comparison_b['drp_chod'] = comparison_b['drp_chod'].fillna(0.0)
comparison_b['drp_finrez'] = comparison_b['drp_finrez'].fillna(0.0)

comparison_b['diff_chod'] = comparison_b['drp_chod'] - comparison_b['excel_chod']
comparison_b['diff_finrez'] = comparison_b['drp_finrez'] - comparison_b['excel_finrez']
comparison_b['total_abs_delta'] = comparison_b['diff_chod'].abs() + comparison_b['diff_finrez'].abs()

comparison_b['reason_flag'] = np.select(
    [
        comparison_b['_merge'] == 'left_only',
        comparison_b['_merge'] == 'right_only',
        (comparison_b['_merge'] == 'both') & (
            (comparison_b['diff_chod'].abs() > 0.01) |
            (comparison_b['diff_finrez'].abs() > 0.01)
        ),
    ],
    ['only_excel', 'only_drp', 'value_mismatch'],
    default='match',
)

comparison_b = comparison_b.drop(columns=['_merge'])
comparison_b = comparison_b.sort_values(['report_month', 'reason_flag', 'total_abs_delta'], ascending=[True, True, False])

print('Первые 50 расхождений (без полностью совпавших записей):')
display(comparison_b[comparison_b['reason_flag'] != 'match'].head(50))

In [ ]:
# Диагностика причин расхождений
reason_share = (
    comparison_b[comparison_b['reason_flag'] != 'match']
    .groupby(['report_month', 'reason_flag'], as_index=False)
    .agg(
        inn_cnt=('inn', 'nunique'),
        rows=('inn', 'size'),
        chod_abs_delta=('diff_chod', lambda s: s.abs().sum()),
        finrez_abs_delta=('diff_finrez', lambda s: s.abs().sum()),
        total_abs_delta=('total_abs_delta', 'sum'),
    )
    .sort_values(['report_month', 'reason_flag'])
)

# Доля вклада каждой причины в суммарное абсолютное расхождение за месяц
reason_share['month_total_abs_delta'] = reason_share.groupby('report_month')['total_abs_delta'].transform('sum')
reason_share['delta_share_pct'] = np.where(
    reason_share['month_total_abs_delta'] != 0,
    reason_share['total_abs_delta'] / reason_share['month_total_abs_delta'] * 100,
    np.nan,
)

display(reason_share)

# Эффект дублирования: строки vs уникальные ИНН
dup_effect = pd.concat(
    [
        excel_raw.groupby('report_month', as_index=False).agg(source=('report_month', lambda _: 'excel'), rows=('inn', 'size'), inn_cnt=('inn', 'nunique')),
        drp_raw.groupby('report_month', as_index=False).agg(source=('report_month', lambda _: 'drp'), rows=('inn', 'size'), inn_cnt=('inn', 'nunique')),
    ],
    ignore_index=True,
)

dup_effect['rows_per_inn'] = np.where(dup_effect['inn_cnt'] != 0, dup_effect['rows'] / dup_effect['inn_cnt'], np.nan)

display(dup_effect.sort_values(['report_month', 'source']))

print('TOP-20 ИНН по |delta_chod| + |delta_finrez|:')
top_delta = comparison_b.sort_values('total_abs_delta', ascending=False)
display(top_delta.head(20))

In [ ]:
# Опционально: сохранить результаты сверки в CSV
save_outputs = True
output_prefix = 'chod_finrez_q1_reconciliation'

if save_outputs:
    summary_a.to_csv(f'{output_prefix}_summary_by_month.csv', index=False, encoding='utf-8-sig')
    comparison_b.to_csv(f'{output_prefix}_diff_by_inn.csv', index=False, encoding='utf-8-sig')
    reason_share.to_csv(f'{output_prefix}_reason_share.csv', index=False, encoding='utf-8-sig')
    print('Файлы сохранены:')
    print(f'- {output_prefix}_summary_by_month.csv')
    print(f'- {output_prefix}_diff_by_inn.csv')
    print(f'- {output_prefix}_reason_share.csv')